In [1]:
import numpy as np
import pandas as pd

import sys
sys.path.append('../')

from src.data_utils import *
from src.tda_utils import *
from src.utils import *

import plotly.express as px
import plotly.graph_objects as go

import glob

In [2]:
# data_name = 'BTCUSDT_1d'
data_name = 'ETHUSDT_1h' # ETHUSDT_1h, SOLUSDT_1h, BTCUSDT_1h
from_date = '2018-12-30 23:00:00' # 1h | 2018-12-31 23:00:00, 2020-12-30 23:00:00
# from_date = '2021-01-01'
until_date = '2021-12-31 00:00:00' # 1h | 2021-12-31 23:00:00, 2022-12-31 00:00:00
# until_date = '2022-12-31'

In [3]:
initial_slice_position = 0
slice_size_in_days = 14 #[14, 21, 30, 45]
num_point_for_1d = (1 * 24) # 1h
# num_point_for_1d = 1
slice_size = slice_size_in_days * num_point_for_1d
column_name = 'processed_log_return_wtmra_0'
save = True

In [4]:
# Parameters for the point cloud creation
dimension = 2
time_delay = 7 #[3, 7]
time_delay_in_days = (time_delay * 24) # 1h
# time_delay_in_days = 7
stride = 1

In [5]:
# Importing data
data = pd.read_csv(f'../data/01-output-{data_name}-from-{from_date}-until-{until_date}-log-return.csv', parse_dates=['date'], index_col='date')

In [6]:
data.head()

,open,high,low,close,volume,original_close,date_ordinal,processed_close,processed_log_return,outliers_processed_log_return,...,processed_log_return_wtmra_5_4,processed_log_return_wtmra_5_4_3,processed_log_return_wtmra_5_4_3_2,processed_log_return_wtmra_5_4_3_2_1,processed_log_return_wtmra_5_4_3_2_1_0,processed_log_return_wtmra_0_1,processed_log_return_wtmra_0_1_2,processed_log_return_wtmra_0_1_2_3,processed_log_return_wtmra_0_1_2_3_4,processed_log_return_wtmra_0_1_2_3_4_5
date,,,,,,,,,,,,,,,,,,,,,
2018-12-31 00:00:00,137.09,138.42,136.38,137.77,24848.72881,137.77,1.546214e+09,137.77,0.004875,0.004875,...,-0.021383,-0.093238,-0.136311,0.057040,0.225206,0.361517,0.318444,0.246589,0.246718,0.225206
2018-12-31 01:00:00,137.81,138.66,135.93,136.56,22817.09906,136.56,1.546218e+09,136.56,-0.008822,-0.008822,...,-0.026196,-0.114300,-0.240550,-0.303541,-0.401395,-0.160845,-0.287095,-0.375200,-0.378975,-0.401395
2018-12-31 02:00:00,136.56,136.60,133.83,134.96,36918.42692,134.96,1.546222e+09,134.96,-0.011786,-0.011786,...,-0.030258,-0.122841,-0.291197,-0.624224,-0.536999,-0.245802,-0.414158,-0.506741,-0.513782,-0.536999
2018-12-31 03:00:00,134.94,135.64,132.01,132.53,30887.85225,132.53,1.546225e+09,132.53,-0.018169,-0.018169,...,-0.032899,-0.115683,-0.259412,-0.465811,-0.829048,-0.569636,-0.713365,-0.796149,-0.805204,-0.829048
2018-12-31 04:00:00,132.55,134.38,132.20,133.60,31911.91720,133.60,1.546229e+09,133.60,0.008041,0.008041,...,-0.034325,-0.095523,-0.136542,0.015795,0.370055,0.506597,0.465578,0.404380,0.394399,0.370055


In [7]:
data.tail()

,open,high,low,close,volume,original_close,date_ordinal,processed_close,processed_log_return,outliers_processed_log_return,...,processed_log_return_wtmra_5_4,processed_log_return_wtmra_5_4_3,processed_log_return_wtmra_5_4_3_2,processed_log_return_wtmra_5_4_3_2_1,processed_log_return_wtmra_5_4_3_2_1_0,processed_log_return_wtmra_0_1,processed_log_return_wtmra_0_1_2,processed_log_return_wtmra_0_1_2_3,processed_log_return_wtmra_0_1_2_3_4,processed_log_return_wtmra_0_1_2_3_4_5
date,,,,,,,,,,,,,,,,,,,,,
2021-12-30 20:00:00,3759.99,3769.10,3747.41,3751.06,7182.0528,3751.06,1.640894e+09,3751.06,-0.002378,-0.002378,...,0.038509,-0.023249,-0.036967,-0.085242,-0.106604,-0.069637,-0.083354,-0.145113,-0.120697,-0.106604
2021-12-30 21:00:00,3751.06,3760.00,3712.07,3724.78,8265.9312,3724.78,1.640898e+09,3724.78,-0.007031,-0.007031,...,0.036516,-0.029173,-0.105327,-0.143634,-0.319465,-0.214139,-0.290293,-0.355981,-0.333738,-0.319465
2021-12-30 22:00:00,3724.46,3742.92,3707.66,3736.93,7095.4022,3736.93,1.640902e+09,3736.93,0.003257,0.003257,...,0.035311,-0.029065,-0.124581,-0.131934,0.151166,0.275747,0.180231,0.115855,0.136787,0.151166
2021-12-30 23:00:00,3736.93,3737.71,3681.33,3703.83,6753.5304,3703.83,1.640905e+09,3703.83,-0.008897,-0.008897,...,0.034293,-0.026364,-0.111887,-0.147038,-0.404847,-0.292961,-0.378483,-0.439140,-0.419305,-0.404847
2021-12-31 00:00:00,3703.84,3717.26,3692.23,3709.27,8297.0767,3709.27,1.640909e+09,3709.27,0.001468,0.001468,...,0.033800,-0.019174,-0.056925,-0.095169,0.069323,0.126248,0.088497,0.035523,0.054767,0.069323


In [8]:
# Plot the reconstructed series
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data[column_name], name='Selected Data'))

fig.update_layout(title='Selected Data', xaxis_title='Date', yaxis_title=column_name)
fig.show()

In [9]:
df = data[column_name]

In [10]:
df.head()

date
2018-12-31 00:00:00    0.168167
2018-12-31 01:00:00   -0.097854
2018-12-31 02:00:00    0.087226
2018-12-31 03:00:00   -0.363237
2018-12-31 04:00:00    0.354260
Name: processed_log_return_wtmra_0, dtype: float64

In [11]:
full_series_with_highlight_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/full_series_with_highlight/'
sliced_time_series_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/sliced_time_series/'
point_cloud_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/point_cloud/'
persistence_diagram_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/persistence_diagram/'
mapper_graph_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/mapper_graph/'
mosaic_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/mosaic/'

features_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/features/'
features_plot_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/features_plot/'
mosaic_and_features_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/mosaic_and_features/'
video_path = f'../results/{column_name}_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}/video/'

In [ ]:
# Create folder for saving results if doesn't exist
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

if save:
    create_folder(full_series_with_highlight_path)
    create_folder(sliced_time_series_path)
    create_folder(point_cloud_path)
    create_folder(persistence_diagram_path)
    create_folder(mapper_graph_path)
    create_folder(mosaic_path)
    create_folder(features_path)
    create_folder(features_plot_path)
    create_folder(mosaic_and_features_path)
    create_folder(video_path)

# code to erase files in folder '../results/sliced_time_series/'

def erase_files(path): 
    files = glob.glob(path + '*')
    for f in files:
        os.remove(f)

if save:
    erase_files(full_series_with_highlight_path)
    erase_files(sliced_time_series_path)
    erase_files(point_cloud_path)
    erase_files(persistence_diagram_path)
    erase_files(mapper_graph_path)
    erase_files(mosaic_path)
    erase_files(features_path)
    erase_files(features_plot_path)
    erase_files(mosaic_and_features_path)
    erase_files(video_path)


In [13]:
df_features = pd.DataFrame()

# Create an empty dataframe with these columns: [initial_slice_position, slice_size, start_date, end_date, connected_components_entropy, loops_entropy, voids_entropy, connected_components_amplitude, loops_amplitude, voids_amplitude, connected_components_number_of_points, loops_number_of_points, voids_number_of_points]
df_features = pd.DataFrame(columns=['initial_slice_position', 'slice_size', 'start_date', 'end_date', 'connected_components_entropy', 'loops_entropy', 'voids_entropy', 'connected_components_amplitude', 'loops_amplitude', 'voids_amplitude', 'connected_components_number_of_points', 'loops_number_of_points', 'voids_number_of_points'])

for i in range(0, (len(data)-slice_size), num_point_for_1d):

    # fig = generate_plot_full_series_with_highlight(df, initial_slice_position + i, slice_size, save, full_series_with_highlight_path, f'full_series_with_highlight_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}', data_name)
    # # fig.show()

    time_series = get_sliced_time_series(df, initial_slice_position + i, slice_size)

    # fig = generate_plot_sliced_time_series(df, initial_slice_position + i, slice_size, save, sliced_time_series_path, f'sliced_time_series_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
    # # fig.show()
    
    point_cloud = create_point_cloud(time_series, dimension, time_delay_in_days, stride)

    # fig = generate_plot_point_cloud(point_cloud, initial_slice_position + i, slice_size, save, point_cloud_path, f'point_cloud_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
    # # fig.show()
    # fig = generate_plot_persistence_diagram(point_cloud, initial_slice_position + i, slice_size, save, persistence_diagram_path, f'persistence_diagram_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
    # # fig.show()
    # fig = generate_plot_mapper_graph(point_cloud, initial_slice_position + i, slice_size, save, mapper_graph_path, f'mapper_graph_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
    # # fig.show()

    entropy_features_dict, amplitude_features_dict, number_of_points_features_dict = get_features(point_cloud)

    # Add the features to the dataframe using concat
    df_features = pd.concat([df_features, pd.DataFrame({
        'initial_slice_position': initial_slice_position + i,
        'slice_size': slice_size,
        'start_date': data.index[initial_slice_position + i],
        'end_date': data.index[initial_slice_position + i + slice_size],
        'connected_components_entropy': entropy_features_dict['connected_components_entropy'],
        'loops_entropy': entropy_features_dict['loops_entropy'],
        'voids_entropy': entropy_features_dict['voids_entropy'],
        'connected_components_amplitude': amplitude_features_dict['connected_components_amplitude'],
        'loops_amplitude': amplitude_features_dict['loops_amplitude'],
        'voids_amplitude': amplitude_features_dict['voids_amplitude'],
        'connected_components_number_of_points': number_of_points_features_dict['connected_components_number_of_points'],
        'loops_number_of_points': number_of_points_features_dict['loops_number_of_points'],
        'voids_number_of_points': number_of_points_features_dict['voids_number_of_points']
    }, index=[0])], ignore_index=True)        

    # full_path_img1 = f'{sliced_time_series_path}/sliced_time_series_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
    # full_path_img2 = f'{full_series_with_highlight_path}/full_series_with_highlight_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
    # full_path_img3 = f'{point_cloud_path}/point_cloud_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
    # full_path_img4 = f'{point_cloud_path}/point_cloud_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}_fixed_axis.png'
    # full_path_img5 = f'{persistence_diagram_path}/persistence_diagram_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
    # full_path_img6 = f'{persistence_diagram_path}/persistence_diagram_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}_fixed_axis.png'
    # full_path_img7 = f'{mapper_graph_path}/mapper_graph_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
    # full_path_mosaic = f'{mosaic_path}/mosaic_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
    
    # generate_mosaic(full_path_img1, full_path_img2, full_path_img3, full_path_img4, full_path_img5, full_path_img6, full_path_img7, full_path_mosaic)

/Users/lcjr86/de Jesus Lallement Dropbox/Luiz Carlos de Jesus Junior/PhD_Universidad_Loyola/repos/NCAA-D-24-04812R2/env/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/lcjr86/de Jesus Lallement Dropbox/Luiz Carlos de Jesus Junior/PhD_Universidad_Loyola/repos/NCAA-D-24-04812R2/env/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/lcjr86/de Jesus Lallement Dropbox/Luiz Carlos de Jesus Junior/PhD_Universidad_Loyola/repos/NCAA-D-24-04812R2/env/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/lcjr86/de Jesus Lallement Dropbox/Luiz Carlos de Jesus Junior/PhD_Universidad_Loyola/repos/NCAA-D-24-04812R2/env/lib/python3.1

In [14]:
# Save the dataframe with the features
if save:
    df_features.to_csv(f'{features_path}/features_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}.csv', index=False)

In [15]:
df_features = pd.read_csv(f'{features_path}/features_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}.csv')

In [16]:
df_features.columns

Index(['initial_slice_position', 'slice_size', 'start_date', 'end_date',
       'connected_components_entropy', 'loops_entropy', 'voids_entropy',
       'connected_components_amplitude', 'loops_amplitude', 'voids_amplitude',
       'connected_components_number_of_points', 'loops_number_of_points',
       'voids_number_of_points'],
      dtype='object')

In [17]:
list_columns_to_normalize = ['connected_components_entropy', 'loops_entropy', 'voids_entropy',
       'connected_components_amplitude', 'loops_amplitude', 'voids_amplitude',
       'connected_components_number_of_points', 'loops_number_of_points',
       'voids_number_of_points'
       ]
df_features_normalized = normalize_dataframe(df_features, list_columns_to_normalize)
df_features_normalized_v2 = normalize_dataframe(df_features, list_columns_to_normalize, (-1, 1))

In [18]:
# df_features_normalized_v2.describe()

In [19]:
# df_features_normalized_v2.head()

In [20]:
# list_columns_to_accumulate = ['connected_components_entropy', 'loops_entropy', 'voids_entropy',
#        'connected_components_amplitude', 'loops_amplitude', 'voids_amplitude',
#        'connected_components_number_of_points', 'loops_number_of_points',
#        'voids_number_of_points'
#        ]
# df_features_normalized_accumulated = accumulate_dataframe(df_features_normalized_v2, list_columns_to_accumulate)

In [21]:
# df_features_normalized_accumulated.head()

In [22]:
# for i in range(0, (len(data)-slice_size), num_point_for_1d):
#     mosaic_path_img = f'{mosaic_path}/mosaic_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
#     features_path_img = f'{features_plot_path}/features_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
#     features_accumulate_path_img = f'{features_plot_path}/features_accumulate_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
#     mosaic_and_features_path_img = f'{mosaic_and_features_path}/mosaic_and_features_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
#     fig = generate_features_plot(data, column_name, df_features_normalized_v2, initial_slice_position + i, slice_size, save, features_plot_path, f'features_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
#     # fig = generate_features_plot(data, column_name, df_features_normalized_accumulated, initial_slice_position + i, slice_size, save, features_plot_path, f'features_accumulate_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
#     generate_mosaic_and_features(mosaic_path_img, features_path_img, mosaic_and_features_path_img)

In [23]:
# create_mosaic_video(f'{mosaic_and_features_path}/', f'{video_path}/', f'mosaic_and_features_video_{slice_size}.mp4')